# 02 — Cluster Filtering and Statistics

Apply quality filters to the RCSB 100%-identity sequence clusters and report statistics.

**Filters applied:**
1. Method == X-ray diffraction
2. Resolution ≤ 2.0 Å (also tested at 2.5 Å)
3. Structure has deposited waters
4. Cluster has ≥ 5 qualifying PDB entries

**Check-in point:** Report these statistics before proceeding to structural alignment.

In [ ]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

sys.path.insert(0, '../scripts')
from cluster_filter import load_manifest_index, filter_clusters, cluster_size_histogram, CONSERVATION_DIR

print('Loading manifest index...')
manifest_index = load_manifest_index()
print(f'Manifest entries: {len(manifest_index):,}')

## 1. Run filtering at both resolution cutoffs

In [ ]:
print('Filtering at 2.0 Å...')
clusters_2A, stats_2A = filter_clusters(
    manifest_index=manifest_index,
    resolution_cutoff=2.0,
    min_cluster_size=5
)
print('Filtering at 2.5 Å...')
clusters_25A, stats_25A = filter_clusters(
    manifest_index=manifest_index,
    resolution_cutoff=2.5,
    min_cluster_size=5
)
print('Done.')

## 2. Filtering funnel — how many entries are dropped at each stage

In [ ]:
def print_funnel(stats, label):
    print(f'=== {label} ===' )
    print(f"Total clusters in RCSB file:        {stats['total_clusters_in_file']:>10,}")
    print(f"Total entity-cluster tokens:        {stats['total_member_tokens']:>10,}")
    print(f"  Dropped (not in npz dataset):     {stats['dropped_not_in_manifest']:>10,}")
    print(f"  Dropped (not X-ray):              {stats['dropped_not_xray']:>10,}")
    print(f"  Dropped (resolution > cutoff):    {stats['dropped_bad_resolution']:>10,}")
    print(f"  Dropped (no waters):              {stats['dropped_no_waters']:>10,}")
    print(f"Clusters with >=1 qualifying PDB:   {stats['clusters_with_any_qualifying']:>10,}")
    print(f"Clusters with >={stats['min_cluster_size']} qualifying PDBs:   {stats['clusters_after_size_filter']:>10,}")
    print()

print_funnel(stats_2A, '2.0 Å cutoff')
print_funnel(stats_25A, '2.5 Å cutoff')

## 3. Cluster size distributions

In [ ]:
# Load pre-computed per-cluster counts (min_cluster_size=1 to see all)
# Re-run with min_cluster_size=1 to get the full distribution
_, stats_2A_full = filter_clusters(
    manifest_index=manifest_index, resolution_cutoff=2.0, min_cluster_size=1
)
_, stats_25A_full = filter_clusters(
    manifest_index=manifest_index, resolution_cutoff=2.5, min_cluster_size=1
)
counts_2A = np.array(stats_2A_full['qualifying_pdb_counts'])
counts_25A = np.array(stats_25A_full['qualifying_pdb_counts'])

print(f'Clusters with >=1 qualifying PDB (2.0A): {len(counts_2A):,}')
print(f'Clusters with >=1 qualifying PDB (2.5A): {len(counts_25A):,}')

In [ ]:
bins = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 10000]
bin_labels = ['1', '2-4', '5-9', '10-19', '20-49', '50-99', '100-199', '200-499', '500-999', '≥1000']

def compute_hist(counts, bins):
    hist, _ = np.histogram(counts, bins=bins)
    return hist

h2A = compute_hist(counts_2A, bins)
h25A = compute_hist(counts_25A, bins)

print('Cluster size   |  2.0 Å  |  2.5 Å')
print('-' * 38)
for label, n2, n25 in zip(bin_labels, h2A, h25A):
    print(f'{label:14s} | {n2:7,} | {n25:7,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, counts, label, color in [
    (axes[0], counts_2A, '2.0 Å cutoff', 'steelblue'),
    (axes[1], counts_25A, '2.5 Å cutoff', 'darkorange'),
]:
    # Log-scale histogram of cluster sizes
    log_bins = np.logspace(0, 4, 30)
    ax.hist(counts, bins=log_bins, color=color, edgecolor='white', alpha=0.85)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.axvline(5, color='red', linestyle='--', linewidth=1.2, label='min_cluster_size=5')
    ax.set_xlabel('Cluster size (# qualifying PDB entries)', fontsize=12)
    ax.set_ylabel('# clusters', fontsize=12)
    ax.set_title(f'Cluster size distribution — {label}', fontsize=12)
    ax.legend()

    # Annotate key counts
    n_ge5  = (counts >= 5).sum()
    n_ge10 = (counts >= 10).sum()
    n_ge20 = (counts >= 20).sum()
    ax.text(0.62, 0.95, f'≥5:  {n_ge5:,}\n≥10: {n_ge10:,}\n≥20: {n_ge20:,}',
            transform=ax.transAxes, va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig(CONSERVATION_DIR / 'cluster_size_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to conservation/cluster_size_distributions.png')

## 4. Top clusters by size — quick look at what they are

In [ ]:
# Show the 10 largest clusters at 2.0A to get a sense of what they are
# (sorted by filtered cluster size)
sorted_2A = sorted(clusters_2A, key=len, reverse=True)

print('Top 10 largest clusters (2.0 Å, ≥5 members):')
print(f"{'Rank':5s}  {'Size':6s}  {'Sample PDB IDs (first 5)'}")
print('-' * 55)
for rank, cluster in enumerate(sorted_2A[:10], 1):
    sample = ', '.join(cluster[:5])
    print(f"{rank:5d}  {len(cluster):6d}  {sample}")

## 5. Summary statistics for check-in

In [ ]:
def summary_stats(counts, label):
    c = np.array(counts)
    n_ge5  = (c >= 5).sum()
    n_ge10 = (c >= 10).sum()
    n_ge20 = (c >= 20).sum()
    total_entries = c[c >= 5].sum()
    print(f'--- {label} ---')
    print(f'  Clusters with >=5 members:  {n_ge5:,}')
    print(f'  Clusters with >=10 members: {n_ge10:,}')
    print(f'  Clusters with >=20 members: {n_ge20:,}')
    print(f'  Total PDB entries in >=5 clusters: {total_entries:,}')
    c5 = c[c >= 5]
    if len(c5):
        print(f'  Median cluster size (>=5): {np.median(c5):.0f}')
        print(f'  Mean cluster size (>=5):   {c5.mean():.1f}')
        print(f'  Max cluster size:          {c5.max()}')
    print()

summary_stats(counts_2A, '2.0 Å cutoff')
summary_stats(counts_25A, '2.5 Å cutoff')

## 6. Note on missing schema fields

The following fields are **absent** from the preprocessed npz/json files:

- **Occupancy** — not stored. If alternate conformations existed, they were collapsed during preprocessing.
- **Alt-loc identifiers** — not stored. Same consequence.
- **Unit cell / space group** — not in npz or JSON records. Crystal-contact filtering (step 7) will require loading original PDB/mmCIF files from the source mirror.

These are not blocking for steps 1–3. The crystal-contact filter is noted as a future dependency on the raw PDB files.